# 🎬 OpenShorts no Google Colab (Aceleração por GPU / CUDA)

Este notebook executa o servidor **OpenShorts** utilizando a GPU do Google Colab com autenticação segura via **Colab Secrets (Opção A)** para clonar o repositório privado `https://github.com/diegofullstackjs/new-open-shorts.git`.

---

### 🔑 Passo Prévio: Configurar Secrets do Colab
No menu lateral esquerdo do Google Colab, clique no ícone da chave (**Secrets 🔑**) e adicione:
1. **`GITHUB_TOKEN`**: Seu token do GitHub (*Settings -> Developer settings -> Personal access tokens (classic)* com escopo **`repo`**).
2. **`GEMINI_API_KEY`**: Sua chave de API do [Google AI Studio](https://aistudio.google.com/).

*Certifique-se de marcar a opção **"Notebook access"** para ambos os secrets.*

### 1. Verificar GPU Ativa (CUDA)

In [ ]:
!nvidia-smi

### 2. Autenticação via Secrets & Clone do Repositório Privado

In [ ]:
import os
from google.colab import userdata

# 1. Carregar chaves dos Secrets do Colab
try:
    github_token = userdata.get('GITHUB_TOKEN')
    gemini_key = userdata.get('GEMINI_API_KEY')
except Exception as e:
    raise RuntimeError("Erro ao acessar Secrets do Colab. Verifique se configurou GITHUB_TOKEN e GEMINI_API_KEY no menu de 🔑 Secrets e ativou o acesso do notebook!") from e

if not github_token:
    raise ValueError("O secret 'GITHUB_TOKEN' não foi encontrado ou está vazio nos Secrets do Colab.")
if not gemini_key:
    raise ValueError("O secret 'GEMINI_API_KEY' não foi encontrado ou está vazio nos Secrets do Colab.")

# Configurar variáveis de ambiente do runtime
os.environ['GEMINI_API_KEY'] = gemini_key
os.environ['WHISPER_DEVICE'] = 'cuda'
os.environ['WHISPER_COMPUTE_TYPE'] = 'float16'
os.environ['MAX_CONCURRENT_JOBS'] = '2'

# Clonar ou atualizar o repositório privado
repo_url = f"https://{github_token}@github.com/diegofullstackjs/new-open-shorts.git"
target_dir = "/content/new-open-shorts"

if not os.path.exists(target_dir):
    print("🚀 Clonando repositório privado new-open-shorts...")
    !git clone {repo_url} {target_dir}
else:
    print("🔄 Atualizando repositório existente...")
    %cd {target_dir}
    !git pull

%cd {target_dir}
print("✅ Repositório pronto e configurado com sucesso!")

### 3. Instalar Dependências do Sistema e Pacotes Python

In [ ]:
# Instalar pacotes de sistema necessários (FFmpeg, fontes CJK para legendas, etc.)
!apt-get update -qq && apt-get install -y -qq ffmpeg fonts-noto fonts-noto-cjk libgl1-mesa-glx

# Instalar dependências Python otimizadas para GPU
!pip install -q -r requirements.txt
!pip install -q pycloudflared pyngrok uvicorn

### 4. Iniciar Túnel Público (Cloudflare Tunnel) & Servidor FastAPI
Gera uma URL pública `.trycloudflare.com` para conectar diretamente com a **Extensão Chrome 1-Click Shortify** e automações de canal.

In [ ]:
from pycloudflared import try_cloudflare
import subprocess
import time

# Iniciar o Cloudflare Tunnel na porta 8000
tunnel_url = try_cloudflare(port=8000)
print('='*75)
print(f'🚀 URL PÚBLICA DA API DO OPENSHORTS: {tunnel_url.tunnel}')
print('👉 Cole esta URL nas configurações da sua Extensão Chrome 1-Click Shortify!')
print('='*75)

# Iniciar o servidor FastAPI
!python -m uvicorn app:app --host 0.0.0.0 --port 8000